# get vmPFC mask

different tries
1. take 'mask_MarsAtlas_mPFC_MNI.nii.gz', create fasaverage_hemi-l/r.gii files do surface based subject specific mask creation (as for NPC masks); merge l/r.nii to one again as last step
* try the freesurfer workaround
2. load destrieux atlas from nilearn and save new .nii file ?!

In [1]:
import argparse
import os
import os.path as op
from nipype.interfaces.freesurfer import SampleToSurface

surf_folder = '/Users/mrenke/data/ds-stressrisk/derivatives/surface_masks'

230911-12:15:03,587 nipype.utils WARNING:
	 A newer version (1.8.4) of nipy/nipype is available. You are using 1.8.3


In [73]:
# 1.1.
hemi = "rh"
in_file = op.join(surf_folder,'mask_MarsAtlas_mPFC_MNI_fs-compatible.nii.gz')
reg_file = '/Users/mrenke/data/ds-stressrisk/derivatives/freesurfer/fsaverage/mri/transforms/reg.mni152.2mm.dat' 
out_file = op.join(surf_folder,f'op.join(surf_folder,mask_MarsAtlas_mPFC_hemi-{hemi}.gii') #.gii

sampler = SampleToSurface(hemi=hemi)
sampler.inputs.source_file = in_file
sampler.inputs.reg_file = reg_file
sampler.inputs.sampling_method = "average"
sampler.inputs.sampling_range = 1
sampler.inputs.sampling_units = "frac"
sampler.inputs.out_file = out_file
sampler.cmdline

# res = sampler.run()
# does not work

# works:
#  mri_vol2surf --hemi lh --src /Users/mrenke/data/ds-stressrisk/derivatives/surface_masks/mask_MarsAtlas_mPFC_MNI_fs-compatible.nii.gz --srcreg /Users/mrenke/data/ds-stressrisk/derivatives/freesurfer/fsaverage/mri/transforms/reg.mni152.2mm.dat --out /Users/mrenke/data/ds-stressrisk/derivatives/surface_masks/mask_MarsAtlas_mPFC_hemi-lh.gii
#  mri_vol2surf --hemi rh --src /Users/mrenke/data/ds-stressrisk/derivatives/surface_masks/mask_MarsAtlas_mPFC_MNI_fs-compatible.nii.gz --srcreg /Users/mrenke/data/ds-stressrisk/derivatives/freesurfer/fsaverage/mri/transforms/reg.mni152.2mm.dat --out /Users/mrenke/data/ds-stressrisk/derivatives/surface_masks/mask_MarsAtlas_mPFC_hemi-rh.gii
# https://surfer.nmr.mgh.harvard.edu/fswiki/mri_vol2surf

'mri_vol2surf --hemi rh --o /Users/mrenke/data/ds-stressrisk/derivatives/surface_masks/mask_MarsAtlas_mPFC_hemi-rh.gii --reg /Users/mrenke/data/ds-stressrisk/derivatives/freesurfer/fsaverage/mri/transforms/reg.mni152.2mm.dat --projfrac-avg 1.000 --mov /Users/mrenke/data/ds-stressrisk/derivatives/surface_masks/mask_MarsAtlas_mPFC_MNI_fs-compatible.nii.gz'

In [ ]:
# 1.2. make .nii file Freesurfer compatible
# issues with  input file ()
# https://github.com/nipy/nibabel/issues/428
in_file =op.join(surf_folder,'mask_MarsAtlas_mPFC_MNI.nii.gz')

import nibabel as nib

img = nib.load(in_file)
print(img.header)
img.shape

import numpy as np
img.header.set_data_dtype(np.float32)
nib.save(img, op.join(surf_folder,'mask_MarsAtlas_mPFC_MNI_fs-compatible.nii.gz'))


In [55]:
# 1.2. 

from nilearn import datasets

atlas = datasets.fetch_atlas_destrieux_2009(legacy_format=False) 
regions = atlas['labels'].copy() # regions as pandas dataframe with 'legacy_format=False'

[region for region in regions['name'] if f'front' in region]

['L G_and_S_frontomargin',
 'L G_and_S_transv_frontopol',
 'L G_front_inf-Opercular',
 'L G_front_inf-Orbital',
 'L G_front_inf-Triangul',
 'L G_front_middle',
 'L G_front_sup',
 'L S_front_inf',
 'L S_front_middle',
 'L S_front_sup',
 'R G_and_S_frontomargin',
 'R G_and_S_transv_frontopol',
 'R G_front_inf-Opercular',
 'R G_front_inf-Orbital',
 'R G_front_inf-Triangul',
 'R G_front_middle',
 'R G_front_sup',
 'R S_front_inf',
 'R S_front_middle',
 'R S_front_sup']

In [76]:
# transform_npc
from nipype.interfaces.freesurfer import SurfaceTransform

def transform_surface(in_file,
        out_file, 
        target_subject,
        hemi,
        source_subject='fsaverage'):

    sxfm = SurfaceTransform(subjects_dir= subjects_dir)
    sxfm.inputs.source_file = in_file
    sxfm.inputs.out_file = out_file
    sxfm.inputs.source_subject = source_subject
    sxfm.inputs.target_subject = target_subject
    sxfm.inputs.hemi = fs_hemi
    # sxfm.cmdline #helps with debugging 
    r = sxfm.run()
    return r

In [104]:
bids_folder = '/Users/mrenke/data/ds-stressrisk'
subject = '01'
roi = 'mpfc'
hemi = 'R'

#
subjects_dir = op.join(bids_folder, 'derivatives', 'freesurfer')

target_subject = f'sub-{subject}'

if hemi == 'R':
    fs_hemi = 'rh'
elif hemi == 'L':
    fs_hemi = 'lh'

in_file = op.join(bids_folder, f'derivatives/surface_masks/mask_MarsAtlas_mPFC_hemi-{fs_hemi}.gii')

target_dir = op.join(bids_folder, 'derivatives', f'{roi}_masks', target_subject)

if not op.exists(target_dir):
    os.makedirs(target_dir)

out_file = op.join(target_dir, f'{target_subject}_space-fsnative_hemi-{fs_hemi}.{roi}.gii')


transform_surface(in_file, out_file, target_subject, hemi)

230911-17:07:53,56 nipype.interface INFO:
	 stdout 2023-09-11T17:07:53.055911:
230911-17:07:53,60 nipype.interface INFO:
	 stdout 2023-09-11T17:07:53.055911:7.2.0
230911-17:07:53,60 nipype.interface INFO:
	 stdout 2023-09-11T17:07:53.055911:
230911-17:07:53,60 nipype.interface INFO:
	 stdout 2023-09-11T17:07:53.055911:setenv SUBJECTS_DIR /Users/mrenke/data/ds-stressrisk/derivatives/freesurfer
230911-17:07:53,60 nipype.interface INFO:
	 stdout 2023-09-11T17:07:53.055911:cd /Users/mrenke/git/stress_risk/stress_risk/fmri_analysis/surface
230911-17:07:53,61 nipype.interface INFO:
	 stdout 2023-09-11T17:07:53.055911:mri_surf2surf --hemi rh --tval /Users/mrenke/data/ds-stressrisk/derivatives/mpfc_masks/sub-01/sub-01_space-fsnative_hemi-rh.mpfc.gii --sval /Users/mrenke/data/ds-stressrisk/derivatives/surface_masks/mask_MarsAtlas_mPFC_hemi-rh.gii --srcsubject fsaverage --trgsubject sub-01 
230911-17:07:53,61 nipype.interface INFO:
	 stdout 2023-09-11T17:07:53.055911:
230911-17:07:53,61 nipype.i

In [81]:
from neuropythy.freesurfer import subject as fs_subject


In [95]:
import argparse
import os
import os.path as op
import numpy as np
from nilearn import surface
from neuropythy.freesurfer import subject as fs_subject
from neuropythy.io import load, save
from neuropythy.mri import (is_image, is_image_spec, image_clear, to_image)

hemi = 'R'

#
ses_anat = 1
target_subject = f'sub-{subject}'

if hemi == 'R':
    fs_hemi = 'rh'

elif hemi == 'L':
    fs_hemi = 'lh'

target_dir = op.join(bids_folder, 'derivatives', f'{roi}_masks', target_subject)

if not op.exists(target_dir):
    os.makedirs(target_dir)

fsnative_fn = op.join(target_dir, f'{target_subject}_space-fsnative_hemi-{fs_hemi}.{roi}.gii')
mask_data = surface.load_surf_data(fsnative_fn).astype(bool)

subjects_dir = op.join(bids_folder, 'derivatives', 'freesurfer')


target_fn = op.join(target_dir, f'sub-{subject}_space-T1w_{roi}_{hemi}.nii.gz')
sub = fs_subject(op.join(bids_folder, 'derivatives', 'freesurfer', f'sub-{subject}'))
im = load(op.join(bids_folder, 'derivatives', 'fmriprep', f'sub-{subject}',f'ses-{ses_anat}',
'anat', f'sub-{subject}_ses-{ses_anat}_desc-preproc_T1w.nii.gz'))
im = to_image(image_clear(im, fill=0.0), dtype=np.int)

print('Generating volume...')

if hemi == 'R':
    data = (np.zeros(sub.lh.vertex_count), mask_data)    
elif hemi == 'L':
    data = (mask_data, np.zeros(sub.rh.vertex_count))
            
new_im = sub.cortex_to_image(data,
        im,
        hemi=None,
        method='nearest',
        fill=0.0)

print('Exporting volume file: %s' % target_fn)
save(target_fn, new_im)
print('surface_to_image complete!')

/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/ipykernel_18894/2027826340.py:37: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  im = to_image(image_clear(im, fill=0.0), dtype=np.int)
/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/neuropythy/mri/images.py:112: FutureWarning: Image data has type int64, which may cause incompatibilities with other tools. This will error in NiBabel 5.0. This warning can be silenced by passing the dtype argument to Nifti1Image().
  img = cls(arr, aff, **kw)


Generating volume...
Exporting volume file: /Users/mrenke/data/ds-stressrisk/derivatives/vmpfc_masks/sub-01/sub-01_space-T1w_vmpfc_R.nii.gz
surface_to_image complete!


In [97]:
new_im.shape

(170, 256, 256)

In [102]:

hemi = 'R'

fsnative_fn_L = op.join(target_dir, f'{target_subject}_space-fsnative_hemi-lh.{roi}.gii')
mask_data_L = surface.load_surf_data(fsnative_fn_L).astype(bool)

fsnative_fn_R = op.join(target_dir, f'{target_subject}_space-fsnative_hemi-rh.{roi}.gii')
mask_data_R = surface.load_surf_data(fsnative_fn_R).astype(bool)

subjects_dir = op.join(bids_folder, 'derivatives', 'freesurfer')

sub = fs_subject(op.join(bids_folder, 'derivatives', 'freesurfer', f'sub-{subject}'))
im = load(op.join(bids_folder, 'derivatives', 'fmriprep', f'sub-{subject}',f'ses-{ses_anat}',
'anat', f'sub-{subject}_ses-{ses_anat}_desc-preproc_T1w.nii.gz'))
im = to_image(image_clear(im, fill=0.0), dtype=np.int)

print('Generating volume...')

data = (mask_data_L, mask_data_R)
            
new_im = sub.cortex_to_image(data,
        im,
        hemi=None,
        method='nearest',
        fill=0.0)

target_fn = op.join(target_dir, f'sub-{subject}_space-T1w_{roi}_hemi-both.nii.gz')
save(target_fn, new_im)

/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/ipykernel_18894/4117237850.py:16: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  im = to_image(image_clear(im, fill=0.0), dtype=np.int)
/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/neuropythy/mri/images.py:112: FutureWarning: Image data has type int64, which may cause incompatibilities with other tools. This will error in NiBabel 5.0. This warning can be silenced by passing the dtype argument to Nifti1Image().
  img = cls(arr, aff, **kw)


Generating volume...
